## Проверка модели

In [1]:
import torch

from ml.models import model_detector, model_detector_code

model_detector.to('cuda')
model_detector.eval()

x1, x2 = model_detector(torch.randn(1, 1, 640, 640).to('cuda'))
print(x1.shape)
print(x2.keys())


CRITICAL:root:.env.prod


DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
    

AttributeError: 'ModuleList' object has no attribute 'anchors'

## Гиперпарамтеры

In [2]:
from ml.config import WORKDIR
from ml.models import model_detector
from ml.loss_function.loss import OCRYOLOv8Loss

from torch.optim import AdamW
from torch.optim.lr_scheduler import MultiStepLR
from datetime import datetime

loss_func = OCRYOLOv8Loss()
opt = AdamW(model_detector.parameters(), lr=0.01, weight_decay=5e-4)
lr_sched = MultiStepLR(opt, milestones=[10, 35, 50], gamma=0.1)

epochs = 1
batch_size_train = 64
batch_size_val = 16
batch_size_test = 32
dataload_workers = 8

threshold_loss = 0.03
early_stopping = 12
models_dir = WORKDIR / 'ml' / 'model_weights' / 'detector' / f'{datetime.now().strftime("%Y%m%d-%H%M%S")}'
models_dir.mkdir(parents=True)


#### Функции для графиков по лоссам и метрикам

In [3]:
from ml.utils import plot_curves


def plot_validation_metrics(history_arg, save_path=None):
    curves = [
        {
            "label": "Val Loss",
            "values": history_arg["val_loss"],
            "color": "red"
        },
        {
            "label": "mAP@0.5",
            "values": history_arg["map50"],
            "color": "blue"
        },
        {
            "label": "mAP@0.5:0.95",
            "values": history_arg["map5095"],
            "color": "green"
        }
    ]

    plot_curves(
        curves=curves,
        title="Validation Loss & Metrics",
        ylabel="Loss / mAP",
        save_path=save_path
    )

def plot_training_dynamics(history_arg, save_path=None):
    curves = [
        {
            "label": "Train Loss",
            "values": history_arg["train_loss"],
            "color": "orange"
        },
        {
            "label": "Val Loss",
            "values": history_arg["val_loss"],
            "color": "red"
        },
        {
            "label": "Learning Rate",
            "values": history_arg["lr"],
            "color": "purple"
        }
    ]

    plot_curves(
        curves=curves,
        title="Training Dynamics",
        ylabel="Loss / LR",
        save_path=save_path
    )


#### Датасет, Даталоадеры

In [4]:
from ml.dataclass_detector import OCRDetectorDataset
from torch.utils.data import DataLoader
from ml.config import WORKDIR


train_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'train', 'train')
val_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'val', 'val')
test_dset = OCRDetectorDataset(WORKDIR / 'dataset' / 'iam-form-stratified' / 'test', 'val')

train_loader = DataLoader(
    dataset=train_dset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)
val_loader = DataLoader(
    dataset=val_dset,
    batch_size=batch_size_val,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)
test_loader = DataLoader(
    dataset=val_dset,
    batch_size=batch_size_test,
    shuffle=True,
    num_workers=dataload_workers,
    collate_fn=OCRDetectorDataset.collate_fn,
    pin_memory=True,
    persistent_workers=True
)

WARNING  | D24-01-2026 T20:29:10 | ml\dataclass_detector.py: def create_data(): line - 163 В наборе данных 960 размеченных изображений
WARNING  | D24-01-2026 T20:29:10 | ml\dataclass_detector.py: def create_data(): line - 163 В наборе данных 240 размеченных изображений
WARNING  | D24-01-2026 T20:29:10 | ml\dataclass_detector.py: def create_data(): line - 163 В наборе данных 326 размеченных изображений


## Обучение & Валидация

In [5]:
import torch
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.metrics import ap_per_class, box_iou
from ml.config import env
from tqdm import tqdm
from ml.models import model_detector

model_detector.to(env.device)


print(f'\033[34mОбучение началось\033[0m | Эпохи: \033[33m{epochs}\033[0m')


train_loss, val_loss, lr_list, metrics, min_loss = [], [], [], [], 1.0
plateau_loss_epochs = 0
iouv = torch.linspace(0.5, 0.95, 10).to(env.device)

for epoch in range(1, epochs + 1):

    model_detector.train()

    last_losses_train = []
    train_loop = tqdm(train_loader, leave=False, desc=f'Training \033[34m#{epoch}\033[0m')
    for img, targets, words in train_loop:
        "Предсказание модели на батч"
        img = img.to(env.device, non_blocking=True)
        targets = [t.to(env.device, non_blocking=True) for t in targets]

        preds = model_detector(img)
        print(type(preds), len(preds), preds.keys())
        print('boxes', preds['boxes'].shape) # Не забыть в чек-нотбуке поменять размерность тензора в preds при тестировании функции потерь. Там рег макс используется, но я его выкинул сейчас
        print('scores', preds['scores'].shape) # Не забыть в чек-нотбуке поменять размерность тензора в preds при тестировании функции потерь. Там рег макс используется, но я его выкинул сейчас

        total_loss_tsr, losses_tsr = loss_func(preds, targets)

        opt.zero_grad()
        total_loss_tsr.backward()
        opt.step()

        "Считаем лосс"
        total_loss = total_loss_tsr.item()
        losses = [(loss_param, item.item()) for loss_param, item in losses_tsr.items()]
        losses.append(total_loss)
        last_losses_train = losses

        train_loss.append(losses)

    print(f"\033[32mTRAINING\033[0m | Epoch {epoch}, train_loss={last_losses_train[-1]:.4f}, {losses[0][0]}_loss={last_losses_train[0][1]:.4f}, {losses[1][0]}_loss={last_losses_train[1][1]:.4f}")
    train_loss.append(last_losses_train)


    model_detector.eval()
    with torch.no_grad():
        last_losses_val = []
        stats = []

        val_loop = tqdm(val_loader, leave=False, desc=f'\033[Validation \033[36m#{epoch}\033[0m')
        for img, targets, words in val_loader:

            img = img.to(env.device, non_blocking=True)
            targets = [t.to(env.device, non_blocking=True) for t in targets]

            preds = model_detector(img)
            total_loss_tsr, losses_tsr = loss_func(preds, targets)

            total_loss = total_loss_tsr.item()
            losses = [(loss_param, item.item()) for loss_param, item in losses_tsr.items()]
            losses.append(total_loss)
            last_losses_val = losses

            preds_nms = non_max_suppression(
                preds,
                conf_thres=0.25,
                iou_thres=0.45,
                max_det=300
            )

            # Метрики
            for i, pred in enumerate(preds_nms):
                gt = targets[i]  # [num_gt, 5] -> xyxy + cls

                nl = gt.shape[0]
                npr = pred.shape[0]

                correct = torch.zeros(npr, len(iouv), dtype=torch.bool, device=env.device)

                if npr == 0:
                    if nl:
                        stats.append((
                            correct,
                            torch.zeros(0, device=env.device),
                            torch.zeros(0, device=env.device),
                            gt[:, 4]
                        ))
                    continue

                if nl:
                    iou = box_iou(pred[:, :4], gt[:, :4])
                    for j in range(len(iouv)):
                        correct[:, j] = (iou >= iouv[j]).any(1)

                stats.append((
                    correct,
                    pred[:, 4],   # confidence
                    pred[:, 5],   # pred class
                    gt[:, 4]      # gt class
                ))

    lr_sched.step()
    lr = lr_sched.get_last_lr()
    lr_list.append(lr)

    "Подсчёт метрик и лоссов за эпоху"
    total_losses = [all_loss[-1] for all_loss in val_loss]
    val_loss_mean = sum(total_losses) / len(total_losses)

    if len(stats):
        stats = [torch.cat(x, 0).cpu().numpy() for x in zip(*stats)]

        if stats[0].any():
            p, r, ap, f1, ap_class = ap_per_class(*stats)
            map50 = ap[:, 0].mean()
            map5095 = ap.mean()
        else:
            map50, map5095 = 0.0, 0.0

    else:
        map50, map5095 = 0.0, 0.0

    print(f"\033[34mVALIDATION\033[0m Epoch {epoch} | val_loss={val_loss_mean:.4f}, {last_losses_val[0][0]}_loss={last_losses_val[0][1]:.4f}, {last_losses_val[1][0]}_loss={last_losses_val[1][1]:.4f}| mAP@0.5={map50:.4f} | mAP@0.5:0.95={map5095:.4f}")

    history = {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "map50": map50,
        "map5095": map5095,
        "lr": lr_list
    }

    if min_loss - min_loss * threshold_loss > last_losses_val[-1]:
        min_loss = last_losses_val[-1]

        checkpoint = {
            'model_detector': model_detector_code,
            'state_model': model_detector.state_dict(),
            'state_opt': opt.state_dict(),
            'state_lr_scheduler': lr_sched.state_dict(),
            'history': history,
            'save_epoch': epoch
        }
        torch.save(checkpoint, models_dir.joinpath(f'model_detector{epoch}.pth'))
        plateau_loss_epochs = 0
        print(f'На эпохе - \033[35m{epoch}\033[0m модель сохранена | Ранняя остановка обучения изменена', end='\n\n')

    "Early Stopping"
    if plateau_loss_epochs >= early_stopping:
        print(f'\033[31m{'!!!' * 10} Принудительная остановка обучения, нет прогресса {'!!!' * 10}\033[0m')
        raise Exception("Early Stopping")

    plateau_loss_epochs += 1

plot_training_dynamics(history, models_dir / 'training.png')
plot_validation_metrics(history, models_dir / 'validation.png')

print(f'{'>>>' * 10} Обучение завершено {'<<<' * 10}')

Обучение началось | Эпохи: 1


Training #1:   0%|          | 0/15 [00:00<?, ?it/s]

<class 'dict'> 3 dict_keys(['boxes', 'scores', 'feats'])
boxes torch.Size([64, 64, 8400])
scores torch.Size([64, 80, 8400])


ValueError: Target size (torch.Size([8400])) must be the same as input size (torch.Size([80, 8400]))